# Holo + Apo Multi-panel Figure 2

This notebook generalizes the mixed holo/apo figure builder so each figure can decide its own panel grid, single-item rows/columns, blank slots, and output file.

In [1]:
import pickle
from copy import deepcopy
from pathlib import Path
from string import ascii_lowercase
ascii_lowercase = list(ascii_lowercase) + ["aa", "ab", "ac", "ad"]

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import rcParams

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR.parents[1]
HOLO_DIR = BASE_DIR.parent / "other_tools"
APO_DIR = BASE_DIR.parent / "other_tools_apos"
OUTPUT_DIR = BASE_DIR
OUTPUT_DIR.mkdir(exist_ok=True)

COLORS = {
    "orange": "#d55e00",
    "green": "#009e73",
    "blue": "#0072b2",
    "yellow": "#f0e442",
}

HOLO_EXTRA_SET = PROJECT_DIR / "training_data/7.Extra_set"
APO_EXTRA_SET = PROJECT_DIR / "training_data/8.Apos/Extra_set"

with open(APO_EXTRA_SET / "features.pkl", "rb") as f:
    apo_news = pickle.load(f)

with open(APO_EXTRA_SET / "apos.pkl", "rb") as f:
    apos = {
        holo: {apo: chains for apo, chains in mapping.items() if apo in apo_news}
        for holo, mapping in pickle.load(f).items()
    }
    
apos

{'8jp0': {'8sgj': ['A']},
 '8f4s': {'7l6r': ['A']},
 '7gqu': {'6yhr': ['A']},
 '7yg5': {'7xlq': ['A']},
 '8v81': {'5uak': ['A']},
 '6z1m': {'6z1h': ['A']},
 '9mfs': {'8szq': ['A']},
 '7f8p': {'5e1i': ['A']},
 '7p2v': {'6gx3': ['D']},
 '7qpn': {'2gk9': ['A']},
 '7u4k': {'2gs3': ['A']},
 '7v39': {'7v37': ['A']},
 '7zpe': {'9di9': ['A']},
 '8cgw': {'5c97': ['A']},
 '8crc': {'2ojx': ['A']},
 '8fpi': {'6uen': ['A']},
 '8qtk': {'3pfv': ['A']},
 '8y6w': {'8y6y': ['A']},
 '9dol': {'4n22': ['A']},
 '9ebs': {'9di2': ['B']},
 '9fsj': {'9fsk': ['A']},
 '9o2m': {'5e97': ['A']},
 '9oul': {'9ouk': ['B']},
 '9prs': {'8yf0': ['A']}}

In [2]:
HOLO_ORTHO_POCKETS = pd.read_pickle(HOLO_DIR / "ortho_pockets.pkl")
APO_ORTHO_POCKETS = pd.read_pickle(APO_DIR / "ortho_pockets.pkl")


def get_model_images_dir(dataset_dir, model_name):
    images_root = Path(dataset_dir) / "images"
    model_dir = images_root / model_name
    if model_dir.is_dir():
        return model_dir
    if model_name == "model5":
        return images_root
    return model_dir


def get_rendered_image_path(images_dir, pdb):
    images_dir = Path(images_dir)
    matches = sorted(images_dir.glob(f"{pdb}_*.png"))
    if matches:
        return matches[0]

    legacy = images_dir / f"{pdb}.png"
    if legacy.is_file():
        return legacy

    raise FileNotFoundError(f"Missing rendered PNG for {pdb} in {images_dir}")


def maybe_get_rendered_image_path(images_dir, pdb):
    try:
        return get_rendered_image_path(images_dir, pdb)
    except FileNotFoundError:
        return None


def list_rendered_image_pdbs(images_dir):
    images_dir = Path(images_dir)
    if not images_dir.is_dir():
        return set()

    return {
        p.name.split("_")[0]
        for p in images_dir.glob("*.png")
        if not p.name.endswith("_raw.png")
    }


plt.rcParams["font.family"] = "Aptos"
rcParams["svg.fonttype"] = "none"

In [3]:
from lxml import etree
import os
import shutil
import subprocess

AUTO_SVG_CLEANUP = True
INKSCAPE_APPIMAGE = OUTPUT_DIR / "Inkscape-1.4.4.AppImage"
INKSCAPE_APPDIR = OUTPUT_DIR / "squashfs-root"
INKSCAPE_RUNNER = INKSCAPE_APPDIR / "AppRun"


def _cleanup_local_name(element):
    return etree.QName(element).localname


def _cleanup_is_background_patch(node):
    if _cleanup_local_name(node) != "g":
        return False
    node_id = node.get("id", "")
    if not node_id.startswith("patch_"):
        return False
    drawable_children = [child for child in node if _cleanup_local_name(child) in {"path", "rect", "polygon"}]
    if len(drawable_children) != 1:
        return False
    drawable = drawable_children[0]
    style = drawable.get("style", "")
    fill = drawable.get("fill")
    stroke = drawable.get("stroke")
    if fill is None:
        fill = None
        for part in style.split(";"):
            if part.strip().startswith("fill:"):
                fill = part.split(":", 1)[1].strip()
                break
    if stroke is None:
        stroke = None
        for part in style.split(";"):
            if part.strip().startswith("stroke:"):
                stroke = part.split(":", 1)[1].strip()
                break
    has_fill = fill not in {None, "none", "None"}
    has_stroke = stroke not in {None, "none", "None"}
    return has_fill and not has_stroke


def _cleanup_remove_background_patches(element):
    for child in list(element):
        if _cleanup_is_background_patch(child):
            element.remove(child)
            continue
        _cleanup_remove_background_patches(child)


def strip_background_patches_in_file(svg_path):
    svg_path = Path(svg_path)
    parser = etree.XMLParser(huge_tree=True)
    root = etree.parse(str(svg_path), parser).getroot()

    # Remove the top-level transparent page patch that sits directly under the
    # matplotlib figure group; this is the "demonic patch" visible in Inkscape.
    for figure in root.xpath('.//svg:g[@id="figure_1"]', namespaces={'svg': SVG_NS}):
        for patch in list(figure.xpath('./svg:g[@id="patch_1"]', namespaces={'svg': SVG_NS})):
            figure.remove(patch)

    _cleanup_remove_background_patches(root)
    etree.ElementTree(root).write(
        str(svg_path),
        pretty_print=True,
        xml_declaration=True,
        encoding='utf-8',
    )
    return svg_path


def _inkscape_env():
    runtime_root = OUTPUT_DIR / '.inkscape-runtime'
    for subdir in ['config', 'cache', 'data', 'tmp']:
        (runtime_root / subdir).mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['HOME'] = str(runtime_root)
    env['XDG_CONFIG_HOME'] = str(runtime_root / 'config')
    env['XDG_CACHE_HOME'] = str(runtime_root / 'cache')
    env['XDG_DATA_HOME'] = str(runtime_root / 'data')
    env['TMPDIR'] = str(runtime_root / 'tmp')
    env['_INKSCAPE_GC'] = 'disable'
    return env


def _svg_has_element_id(svg_path, element_id):
    parser = etree.XMLParser(huge_tree=True)
    root = etree.parse(str(svg_path), parser).getroot()
    return bool(root.xpath(f'.//*[@id="{element_id}"]'))


def run_inkscape_fit_to_content(svg_path, plain_svg=False, use_xvfb=True):
    svg_path = Path(svg_path)
    if not INKSCAPE_RUNNER.exists():
        raise FileNotFoundError(
            f"Inkscape runner not found at {INKSCAPE_RUNNER}. Extract the AppImage first."
        )

    cmd = [str(INKSCAPE_RUNNER)]
    if use_xvfb and shutil.which('xvfb-run'):
        cmd = ['xvfb-run', '-a', *cmd]

    if _svg_has_element_id(svg_path, 'figure_1'):
        selection_action = 'select-by-id:figure_1'
    else:
        selection_action = 'select-all:all'

    export_actions = [
        selection_action,
        'page-fit-to-selection',
        f'export-filename:{svg_path}',
        'export-overwrite',
        'export-type:svg',
    ]
    if plain_svg:
        export_actions.append('export-plain-svg')
    export_actions.append('export-do')

    result = subprocess.run(
        [
            *cmd,
            '--batch-process',
            '--with-gui',
            '--vacuum-defs',
            f"--actions={';'.join(export_actions)}",
            str(svg_path),
        ],
        check=False,
        capture_output=True,
        text=True,
        env=_inkscape_env(),
        cwd=str(svg_path.parent),
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Inkscape cleanup failed for {svg_path}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )
    return svg_path


def cleanup_svg_for_assembly(svg_path, plain_svg=False, strip_patches=True, use_xvfb=True):
    svg_path = Path(svg_path)
    if svg_path.suffix.lower() != '.svg' or not AUTO_SVG_CLEANUP:
        return svg_path
    if strip_patches:
        strip_background_patches_in_file(svg_path)
    run_inkscape_fit_to_content(svg_path, plain_svg=plain_svg, use_xvfb=use_xvfb)
    print(f"Cleaned {svg_path}")
    return svg_path


def cleanup_svg_batch(svg_paths, plain_svg=False, strip_patches=True, use_xvfb=True):
    return [
        cleanup_svg_for_assembly(
            svg_path,
            plain_svg=plain_svg,
            strip_patches=strip_patches,
            use_xvfb=use_xvfb,
        )
        for svg_path in map(Path, svg_paths)
    ]


def export_png(svg_path, use_xvfb=True):
    svg_path = Path(svg_path)
    if not INKSCAPE_RUNNER.exists():
        raise FileNotFoundError(
            f"Inkscape runner not found at {INKSCAPE_RUNNER}. Extract the AppImage first."
        )

    env = _inkscape_env()
    env["XDG_DATA_HOME"] = str(Path.home() / ".local/share") # different from trimming/cleaning to recognize Aptos font

    cmd = [str(INKSCAPE_RUNNER)]
    if use_xvfb and shutil.which('xvfb-run'):
        cmd = ['xvfb-run', '-a', *cmd]

    export_actions = [
        # f'export-filename:{svg_path.with_suffix(".png").name}',
        # 'export-overwrite',
        # 'export-type:png',
        # 'export-dpi:600',
        # 'export-do',
        "--export-type=png",
        "--export-dpi=600",
        "--export-filename", svg_path.with_suffix(".png").name,
    ]

    result = subprocess.run(
        [
            *cmd,
            # '--batch-process',
            # '--with-gui',
            # '--vacuum-defs',
            # f"--actions={';'.join(export_actions)}",
            *export_actions,
            str(svg_path.name),
        ],
        check=False,
        capture_output=True,
        text=True,
        env=env,
        cwd=str(svg_path.parent),
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Inkscape cleanup failed for {svg_path}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )

    print(f"Exported {svg_path.with_suffix('.png')}")
    return #svg_path.with_suffix(".png")

In [4]:
holo_models = pd.read_pickle(HOLO_DIR / "models.pkl")
apo_models = pd.read_pickle(APO_DIR / "models_lenient_labelling.pkl")

holo_df = holo_models["model5"]["labelled"]#build_model5_labelled(holo_models)#, excluded_pdbs=("9dnm",))
apo_df = apo_models["model5"]["labelled"]#build_model5_labelled(apo_models)#, excluded_pdbs=("4jqi",))

holo_df.sort_values(["pdb", "prob"], ascending=[True, False], inplace=True)
apo_df.sort_values(["pdb", "prob"], ascending=[True, False], inplace=True)

HOLO_AVAILABLE_PDBS = set(holo_df["pdb"])
APO_AVAILABLE_PDBS = set(apo_df["pdb"])
HOLO_MODEL5_IMAGE_PDBS = list_rendered_image_pdbs(get_model_images_dir(HOLO_DIR, "model5"))
APO_MODEL5_IMAGE_PDBS = list_rendered_image_pdbs(get_model_images_dir(APO_DIR, "model5"))

## Scatter

In [5]:
BASE_SCATTER_CONFIG = {
    "point_size": 125,  # Marker size for the scatter points.
    "jitter_width": 0.2,  # Maximum horizontal spread of the beeswarm offsets.
    "grey_alpha": 0.3,  # Opacity used for points in the background grey class.
    "axis_linewidth": 0.8,  # Line width of the visible y-axis spine.
    "tick_fontsize": 11.5,  # Font size of the y-axis tick labels.
    "label_fontsize": 13.5,  # Font size of the max-score numeric annotation.
    "xlabel_fontsize": 11.8,  # Font size of the '(n = ...)' label below each scatterplot.
    "score_title_fontsize": 13,  # Font size of the 'AlloPockets score' title above each scatterplot.
    "score_title_y": 1.12,  # Vertical offset of the scatterplot title in x-axis coordinates.
    "annotation_x_hoffset": -0.375,  # Horizontal offset multiplier used for the max-score annotation.
    "annotation_x_voffset": 0.125,  # Vertical offset used for the max-score annotation.
    "yticks": [0.0, 0.25, 0.5, 0.75, 1.0],  # Explicit y-tick locations used on every scatterplot.
    "ylabel_right_spine_position": 0.8125,  # Axes-relative x-position of the visible right-hand y-axis spine.
    "ylabel_left_spine_position": 0.1875,  # Axes-relative x-position of the visible left-hand y-axis spine.
    "light_grey": "0.8",  # Color used for points that are neither predicted nor labelled.
    "xlabel_color": "0.35",  # Text color of the '(n = ...)' label below each scatterplot.
}

DATASET_SCATTER_CONFIGS = {
    "holo": {
        **BASE_SCATTER_CONFIG,
        "special_nbins": {"8crc": 1, "7u4k": 1},  # Per-PDB override for the beeswarm bin count.
        "blue_x_shift_by_pdb": {"9mfs": -0.05},  # Per-PDB x-shift for blue-only points.
        "ortho_pockets": HOLO_ORTHO_POCKETS,  # Mapping of holo PDBs to orthosteric-overlap pockets.
        "ortho_skip_pdbs": {"7gqu", "9prs"},  # Holo PDBs where orthosteric highlighting is suppressed.
    },
    "apo": {
        **BASE_SCATTER_CONFIG,
        "special_nbins": {"9di9": 1},  # Per-PDB override for the beeswarm bin count.
        "blue_x_shift_by_pdb": {"9di9": -0.125},  # Per-PDB x-shift for blue-only points.
        "ortho_pockets": APO_ORTHO_POCKETS,  # Mapping of apo PDBs to orthosteric-overlap pockets.
        "ortho_skip_pdbs": {"6yhr", "2gk9", "8yf0", "5c97"},  # Apo PDBs where orthosteric highlighting is suppressed.
    },
}

In [6]:
def get_beeswarm_offsets(probs, pdb_id, jitter_width, special_nbins=None):
    probs = np.asarray(probs, dtype=float)
    if probs.size == 0 or np.nanmax(probs) == np.nanmin(probs):
        return np.zeros_like(probs)

    nbins = (special_nbins or {}).get(pdb_id, 50)
    rng = np.random.default_rng(0)
    quant = np.round(
        nbins * (probs - np.nanmin(probs)) / (np.nanmax(probs) - np.nanmin(probs) + 1e-8)
    )
    order = np.argsort(quant + rng.normal(0, 1e-6, len(probs)))

    x_vals = np.zeros(len(probs))
    layer = 0
    last_bin = None
    for idx in order:
        if quant[idx] != last_bin:
            layer = 0
            last_bin = quant[idx]

        if layer == 0:
            offset = 0
        else:
            offset = (layer + 1) // 2
            if layer % 2 == 0:
                offset = -offset

        x_vals[idx] = offset
        layer += 1

    max_abs = np.max(np.abs(x_vals))
    if max_abs > 0:
        x_vals = x_vals / max_abs * jitter_width
    return x_vals


def get_scatter_point_color(row, config):
    ortho_pockets = config["ortho_pockets"]
    if row["pdb"] in ortho_pockets and row["pdb"] not in config["ortho_skip_pdbs"]:
        if row["pocket"] in ortho_pockets[row["pdb"]]:
            return COLORS["yellow"]
    if row["pred"] and row["label"]:
        return COLORS["green"]
    if row["label"]:
        return COLORS["orange"]
    if row["pred"]:
        return COLORS["blue"]
    return config["light_grey"]


def plot_scatter_panel(pdb_id, df, ax, config, show_xlabel=True, show_score_title=True, y_axis_side="right"):
    sub = df.query("pdb == @pdb_id").copy()
    probs = sub["prob"].to_numpy(dtype=float)
    if len(probs) == 0 or np.all(np.isnan(probs)):
        raise ValueError(f"No valid probabilities found for {pdb_id}")

    sc_colors = np.array([get_scatter_point_color(row, config) for _, row in sub.iterrows()], dtype=object)
    alphas = np.array([
        config["grey_alpha"] if color == config["light_grey"] else 1
        for color in sc_colors
    ])
    x_vals = get_beeswarm_offsets(
        probs,
        pdb_id=pdb_id,
        jitter_width=config["jitter_width"],
        special_nbins=config["special_nbins"],
    )

    grey_mask = sc_colors == config["light_grey"]
    blue_mask = sc_colors == COLORS["blue"]
    color_mask = ~(grey_mask | blue_mask)

    ax.scatter(
        x_vals[grey_mask], probs[grey_mask], s=config["point_size"], c=sc_colors[grey_mask],
        alpha=alphas[grey_mask], edgecolors="none", zorder=1, clip_on=False,
    )
    if np.any(color_mask):
        ax.scatter(
            x_vals[color_mask], probs[color_mask], s=config["point_size"], c=sc_colors[color_mask],
            alpha=alphas[color_mask], edgecolors="none", zorder=2, clip_on=False,
        )

    blue_shift = config["blue_x_shift_by_pdb"].get(pdb_id, 0.0)
    if np.any(blue_mask):
        ax.scatter(
            x_vals[blue_mask] + blue_shift, probs[blue_mask], s=config["point_size"], c=sc_colors[blue_mask],
            alpha=alphas[blue_mask], edgecolors="none", zorder=3, clip_on=False,
        )

    idx_max = int(np.nanargmax(probs))
    text_val = f"{probs[idx_max]:.3f}".rstrip("0").rstrip(".")
    if y_axis_side == "left":
        annotation_x = x_vals[idx_max] + config["jitter_width"] * config["annotation_x_hoffset"] + blue_shift
        annotation_ha = "left"
    else:
        annotation_x = x_vals[idx_max] - config["jitter_width"] * config["annotation_x_hoffset"] + blue_shift
        annotation_ha = "right"
        
    ax.text(
        annotation_x,
        probs[idx_max] + config["annotation_x_voffset"],
        text_val,
        ha=annotation_ha,
        va="center",
        fontsize=config["label_fontsize"],
    )

    ax.set_xlim(-0.5, 0.5)
    ax.set_ylim(0, 1)
    ax.spines["top"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.set_xticks([])
    ax.tick_params(axis="x", which="both", length=0)
    ax.set_yticks(config["yticks"])
    ax.set_yticklabels(["0", "", "0.5", "", "1"])
    ax.tick_params(axis="y", labelsize=config["tick_fontsize"])
    ax.set_ylabel("")

    if y_axis_side == "left":
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(True)
        ax.spines["left"].set_linewidth(config["axis_linewidth"])
        ax.spines["left"].set_position(("axes", config["ylabel_left_spine_position"]))
        ax.yaxis.tick_left()
        ax.yaxis.set_label_position("left")
    else:
        ax.spines["left"].set_visible(False)
        ax.spines["right"].set_visible(True)
        ax.spines["right"].set_linewidth(config["axis_linewidth"])
        ax.spines["right"].set_position(("axes", config["ylabel_right_spine_position"]))
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")

    if show_score_title:
        ax.text(
            0,
            config["score_title_y"],
            "AlloPockets\nscore",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            multialignment="center",
            fontsize=config["score_title_fontsize"],
        )

    ax.set_xlabel(
        f"(n = {len(sub)})" if show_xlabel else "",
        fontsize=config["xlabel_fontsize"],
        color=config["xlabel_color"],
    )
    if show_xlabel:
        ax.xaxis.set_label_coords(0.52, -0.075)

    ax.set_facecolor("none")
    ax.figure.patch.set_alpha(0.0)

## Figure

In [7]:
FIGURE_STYLE = {
    "font_scale": 1.5,  # Global multiplier applied to all figure-level and scatterplot font sizes.
    "show_score_title": False,  # Whether to draw the 'AlloPockets score' title above each scatterplot.
    "left": 0.015,#0.035,  # Left outer figure margin used by the top-level GridSpec.
    "right": 0.985,  # Right outer figure margin used by the top-level GridSpec.
    "top": 0.995,#0.99,  # Top outer figure margin used by the top-level GridSpec.
    "bottom": 0.005,#0.02,  # Bottom outer figure margin used by the top-level GridSpec.
    "outer_hspace": 0.14,  # Vertical spacing between major panel rows.
    "column_wspace": 0.2,  # Horizontal spacing between panel columns within each row.
    "panel_header_ratio": 0.12,  # Fraction of a panel's height reserved for the optional panel title line.
    "panel_hspace": 0.01,  # Vertical spacing between the optional panel title and the panel body.
    "item_group_wspace": -0.025,  # Horizontal spacing between the holo and apo item groups inside a double-item panel.
    "item_title_ratio": 0.12,  # Fraction of an item group's height reserved for the per-structure PDB label.
    "item_hspace": -0.05,#.005,  # Vertical spacing between the per-structure PDB label and its image/scatter pair when the score title is shown.
    "item_hspace_no_score_title": 0.0,  # Vertical spacing between the per-structure PDB label and its image/scatter pair when the score title is hidden.
    "pair_wspace": 0.02,  # Horizontal spacing between the PNG area and the scatterplot area inside one item group.
    "pair_png_ratio": 0.7,  # Relative width allocated to the rendered structure PNG inside one item group.
    "pair_scatter_ratio": 0.3,  # Relative width allocated to the scatterplot inside one item group.
    "scatter_inset": [0.15, 0.21, 0.75, 0.62],  # [x0, y0, width, height] of the scatter inset inside its host axis.
    "protein_max_width": 1.0,  # Maximum normalized width occupied by a PNG inside its image slot.
    "protein_max_height": 1.0,  # Maximum normalized height occupied by a PNG inside its image slot.
    "panel_facecolor": "none",  # Background color of panel helper axes.
    "figure_facecolor": "none",  # Background color of the overall figure canvas.
    "header_fontsize": 15.5,  # Font size of the optional panel-level title line.
    "header_fontweight": "bold",  # Font weight of the optional panel-level title line.
    "item_label_fontsize": 15.0,  # Font size of the per-structure PDB labels above each holo/apo item.
    "item_label_fontweight": "bold",  # Font weight of the per-structure PDB labels above each holo/apo item.
}

In [8]:
def make_scaled_scatter_configs(font_scale):
    scatter_configs = {
        name: {
            **cfg,
            "special_nbins": dict(cfg["special_nbins"]),
            "blue_x_shift_by_pdb": dict(cfg["blue_x_shift_by_pdb"]),
            "ortho_pockets": dict(cfg["ortho_pockets"]),
            "ortho_skip_pdbs": set(cfg["ortho_skip_pdbs"]),
        }
        for name, cfg in DATASET_SCATTER_CONFIGS.items()
    }
    for cfg in scatter_configs.values():
        for key in ["tick_fontsize", "label_fontsize", "xlabel_fontsize", "score_title_fontsize"]:
            cfg[key] *= font_scale
    return scatter_configs


def make_scaled_figure_style():
    style = deepcopy(FIGURE_STYLE)
    style["header_fontsize"] *= style["font_scale"]
    style["item_label_fontsize"] *= style["font_scale"]
    return style


def draw_centered_image(ax, image_path, max_width=0.94, max_height=0.94):
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Missing image asset: {image_path}")

    img = mpimg.imread(image_path)
    height, width = img.shape[:2]
    aspect = width / height

    disp_width = max_width
    disp_height = disp_width / aspect
    if disp_height > max_height:
        disp_height = max_height
        disp_width = disp_height * aspect

    x0 = (1 - disp_width) / 2
    x1 = x0 + disp_width
    y0 = (1 - disp_height) / 2
    y1 = y0 + disp_height

    ax.imshow(
        img,
        extent=(x0, x1, y0, y1),
        aspect="auto",
        interpolation="none",
        resample=False,
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_facecolor("none")
    ax.axis("off")


def get_item_hspace(figure_style):
    if figure_style["show_score_title"] and figure_style["content_mode"] == "image_scatter":
        return figure_style["item_hspace"]
    return figure_style["item_hspace_no_score_title"]


def draw_blank_item_group(fig, subspec, figure_style, item=None):
    group = subspec.subgridspec(
        2,
        1,
        height_ratios=[figure_style["item_title_ratio"], 1 - figure_style["item_title_ratio"]],
        hspace=get_item_hspace(figure_style),
    )
    blank_title_ax = fig.add_subplot(group[0])
    blank_title_ax.set_facecolor(figure_style["panel_facecolor"])
    blank_title_ax.axis("off")
    if item is not None and item.get("display_id"):
        blank_title_ax.text(
            0.5,
            0.55,
            item["display_id"],
            fontsize=figure_style["item_label_fontsize"],
            fontweight=figure_style["item_label_fontweight"],
            ha="center",
            va="center",
        )
    blank_ax = fig.add_subplot(group[1])
    blank_ax.set_facecolor(figure_style["panel_facecolor"])
    blank_ax.axis("off")


def draw_item_group(fig, subspec, item, figure_style, panel_label=None):
    if item.get("is_blank"):
        draw_blank_item_group(fig, subspec, figure_style, item=item)
        return

    image_on_right = item.get("image_on_right", False)

    group = subspec.subgridspec(
        2,
        1,
        height_ratios=[figure_style["item_title_ratio"], 1 - figure_style["item_title_ratio"]],
        hspace=get_item_hspace(figure_style),
    )

    item_title_ax = fig.add_subplot(group[0])
    item_title_ax.set_facecolor(figure_style["panel_facecolor"])
    item_title_ax.axis("off")

    if figure_style["content_mode"] == "image_scatter":
        total_width = figure_style["pair_png_ratio"] + figure_style["pair_scatter_ratio"]
        if image_on_right:
            label_x = 1 - figure_style["pair_png_ratio"] / total_width / 2
            y_axis_side = "left"
        else:
            label_x = figure_style["pair_png_ratio"] / total_width / 2
            y_axis_side = "right"
    else:
        label_x = 0.5
        y_axis_side = None

    item_label = item["display_id"] if panel_label is None else f"{panel_label} {item['display_id']}"
    item_title_ax.text(
        label_x,
        0.55,
        item_label,
        fontsize=figure_style["item_label_fontsize"],
        fontweight=figure_style["item_label_fontweight"],
        ha="center",
        va="center",
    )

    if figure_style["content_mode"] == "image_only":
        image_ax = fig.add_subplot(group[1])
        image_ax.set_facecolor(figure_style["panel_facecolor"])
        if not item.get("missing_image", False):
            draw_centered_image(
                image_ax,
                item["image_path"],
                max_width=figure_style["protein_max_width"],
                max_height=figure_style["protein_max_height"],
            )
        else:
            image_ax.axis("off")
        return

    if image_on_right:
        width_ratios = [figure_style["pair_scatter_ratio"], figure_style["pair_png_ratio"]]
        scatter_idx, image_idx = 0, 1
    else:
        width_ratios = [figure_style["pair_png_ratio"], figure_style["pair_scatter_ratio"]]
        image_idx, scatter_idx = 0, 1

    pair = group[1].subgridspec(
        1,
        2,
        width_ratios=width_ratios,
        wspace=figure_style["pair_wspace"],
    )

    image_ax = fig.add_subplot(pair[image_idx])
    image_ax.set_facecolor(figure_style["panel_facecolor"])
    if not item.get("missing_image", False):
        draw_centered_image(
            image_ax,
            item["image_path"],
            max_width=figure_style["protein_max_width"],
            max_height=figure_style["protein_max_height"],
        )
    else:
        image_ax.axis("off")

    scatter_host_ax = fig.add_subplot(pair[scatter_idx])
    scatter_host_ax.set_facecolor(figure_style["panel_facecolor"])
    scatter_host_ax.axis("off")
    scatter_ax = scatter_host_ax.inset_axes(figure_style["scatter_inset"])
    scatter_ax.set_facecolor("none")
    plot_scatter_panel(
        item["pdb_id"],
        item["df"],
        ax=scatter_ax,
        config=item["scatter_config"],
        show_xlabel=True,
        show_score_title=figure_style["show_score_title"],
        y_axis_side=y_axis_side,
    )


def draw_panel(fig, subspec, panel_label, panel_title, items, figure_style, show_panel_title):
    if show_panel_title:
        panel = subspec.subgridspec(
            2,
            1,
            height_ratios=[figure_style["panel_header_ratio"], 1 - figure_style["panel_header_ratio"]],
            hspace=figure_style["panel_hspace"],
        )
        header_ax = fig.add_subplot(panel[0])
        header_ax.set_facecolor(figure_style["panel_facecolor"])
        header_ax.axis("off")
        header_ax.text(
            0.0,
            0.55,
            f"{panel_label} {panel_title}",
            fontsize=figure_style["header_fontsize"],
            fontweight=figure_style["header_fontweight"],
            ha="left",
            va="top",
        )
        body_spec = panel[1]
    else:
        label_ax = fig.add_subplot(subspec)
        label_ax.set_facecolor("none")
        label_ax.axis("off")
        label_ax.text(
            0.0,
            1.0,
            panel_label,
            fontsize=figure_style["header_fontsize"],
            fontweight=figure_style["header_fontweight"],
            ha="left",
            va="top",
            zorder=10,
        )
        body_spec = subspec

    body = body_spec.subgridspec(1, len(items), wspace=figure_style["item_group_wspace"])
    for idx, item in enumerate(items):
        draw_item_group(
            fig,
            body[idx],
            item,
            figure_style,
            panel_label=None,
        )


# def choose_apo_representative(holo_pdb):
#     candidates = list(apos.get(holo_pdb, {}))
#     if not candidates:
#         return None
#     return candidates[0]


def build_holo_item(holo_pdb, model_name, scatter_configs, needs_scatter):
    holo_images_dir = get_model_images_dir(HOLO_DIR, model_name)
    image_path = maybe_get_rendered_image_path(holo_images_dir, holo_pdb)

    item = {
        "display_id": holo_pdb,
        "image_path": image_path,
        "missing_image": image_path is None,
        "image_on_right": True,
    }
    if needs_scatter:
        item.update({
            "pdb_id": holo_pdb,
            "df": holo_df,
            "scatter_config": scatter_configs["holo"],
        })
    return item


def build_apo_item(apo_pdb, model_name, scatter_configs, needs_scatter):
    apo_images_dir = get_model_images_dir(APO_DIR, model_name)
    image_path = maybe_get_rendered_image_path(apo_images_dir, apo_pdb)

    item = {
        "display_id": apo_pdb,
        "image_path": image_path,
        "missing_image": image_path is None,
        "image_on_right": False,
    }
    if needs_scatter:
        item.update({
            "pdb_id": apo_pdb,
            "df": apo_df,
            "scatter_config": scatter_configs["apo"],
        })
    return item


def build_blank_item(display_id=None):
    item = {"is_blank": True}
    if display_id is not None:
        item["display_id"] = display_id
    return item


def is_single_item_position(row_1b, col_1b, single_item_rows, single_item_cols):
    return row_1b in single_item_rows or col_1b in single_item_cols


def make_panel_data(holo_pdb, is_single_item, scatter_configs, model_name):
    needs_scatter = model_name == "model5"
    holo_item = build_holo_item(holo_pdb, model_name, scatter_configs, needs_scatter)

    if is_single_item:
        return {
            "panel_title": holo_pdb,
            "items": [holo_item],
        }

    apo_pdb = tuple(apos.get(holo_pdb, (None,)))[0] # choose_apo_representative(holo_pdb)
    apo_item = build_blank_item() if apo_pdb is None else build_apo_item(apo_pdb, model_name, scatter_configs, needs_scatter)

    panel_title = f"{holo_pdb} | {apo_pdb}" if apo_pdb is not None else holo_pdb
    return {
        "panel_title": panel_title,
        "items": [holo_item, apo_item],
    }


def make_holo_apo_figure(
    holo_pdbs,
    rows,
    columns,
    figsize,
    output_filename,
    model_name="model5",
    single_item_rows=(),
    single_item_cols=(),
    empty_panel_slots=(),
    show_panel_title=False,
    label_start="a",
):
    single_item_rows = set(single_item_rows)
    single_item_cols = set(single_item_cols)
    empty_panel_slots = set(empty_panel_slots)

    if rows < 1 or columns < 1:
        raise ValueError("rows and columns must both be at least 1")
    if len(holo_pdbs) > rows * columns - len(empty_panel_slots):
        raise ValueError("Not enough panel positions for the requested holo list and empty slots")
    if min(empty_panel_slots, default=1) < 1 or max(empty_panel_slots, default=1) > rows * columns:
        raise ValueError("empty_panel_slots must use 1-based indices within the figure grid")
    if label_start not in ascii_lowercase:
        raise ValueError("label_start must be one lowercase ASCII letter: 'a' through 'z'")

    needs_scatter = model_name == "model5"
    label_start_idx = ascii_lowercase.index(label_start)
    panel_count = len(holo_pdbs)
    if label_start_idx + panel_count > len(ascii_lowercase):
        raise ValueError("Not enough lowercase ASCII letters for the requested panels and label_start")

    if needs_scatter:
        unknown_holos = [pdb for pdb in holo_pdbs if pdb not in HOLO_AVAILABLE_PDBS]
        if unknown_holos:
            raise ValueError(f"Missing holo prediction data for: {unknown_holos}")

    figure_style = make_scaled_figure_style()
    figure_style["figsize"] = figsize
    figure_style["output_path"] = OUTPUT_DIR / output_filename
    figure_style["content_mode"] = "image_scatter" if needs_scatter else "image_only"
    scatter_configs = make_scaled_scatter_configs(figure_style["font_scale"])

    panel_slots = []
    holo_iter = iter(holo_pdbs)
    total_slots = rows * columns
    for slot_1b in range(1, total_slots + 1):
        row_1b = (slot_1b - 1) // columns + 1
        col_1b = (slot_1b - 1) % columns + 1
        is_single = is_single_item_position(row_1b, col_1b, single_item_rows, single_item_cols)
        panel_units = 1 if is_single else 2

        if slot_1b in empty_panel_slots:
            panel_data = None
        else:
            holo_pdb = next(holo_iter, None)
            panel_data = None if holo_pdb is None else make_panel_data(holo_pdb, is_single, scatter_configs, model_name)

        panel_slots.append({
            "slot": slot_1b,
            "row": row_1b,
            "col": col_1b,
            "panel_units": panel_units,
            "panel_data": panel_data,
        })

    if next(holo_iter, None) is not None:
        raise ValueError("Internal error: not all holo PDBs were placed")

    fig = plt.figure(figsize=figure_style["figsize"], facecolor=figure_style["figure_facecolor"])
    row_grid = fig.add_gridspec(
        rows,
        1,
        left=figure_style["left"],
        right=figure_style["right"],
        top=figure_style["top"],
        bottom=figure_style["bottom"],
        hspace=figure_style["outer_hspace"],
    )

    panel_counter = 0
    for row_1b in range(1, rows + 1):
        row_slots = [slot for slot in panel_slots if slot["row"] == row_1b]
        width_ratios = [slot["panel_units"] for slot in row_slots]
        row_subgrid = row_grid[row_1b - 1].subgridspec(
            1,
            columns,
            width_ratios=width_ratios,
            wspace=figure_style["column_wspace"],
        )

        for col_1b, slot in enumerate(row_slots, start=1):
            panel_spec = row_subgrid[0, col_1b - 1]
            panel_data = slot["panel_data"]
            if panel_data is None:
                empty_ax = fig.add_subplot(panel_spec)
                empty_ax.set_facecolor(figure_style["panel_facecolor"])
                empty_ax.axis("off")
                continue

            panel_label = f"{ascii_lowercase[label_start_idx + panel_counter]})"
            draw_panel(
                fig,
                panel_spec,
                panel_label,
                panel_data["panel_title"],
                panel_data["items"],
                figure_style,
                show_panel_title=show_panel_title,
            )
            panel_counter += 1

    fig.savefig(figure_style["output_path"], transparent=True)
    # plt.show()
    plt.close(fig)
    cleanup_svg_for_assembly(figure_style["output_path"])
    print(f"Saved figure to {figure_style['output_path']}")
    # return fig

# Figure 1

# Figure 2

# Tests

# Smaller figs

## Old

## New

## Other models

#### AllositePro

# Auto Assembly

In [9]:
from copy import deepcopy as _deepcopy
from lxml import etree
import re
import subprocess
import shutil
import os


SVG_NS = "http://www.w3.org/2000/svg"
XLINK_NS = "http://www.w3.org/1999/xlink"
NS = {"svg": SVG_NS}
TEXT_LABEL_RE = re.compile(r"^[a-z]{1,2}\)$")
TRANSFORM_RE = re.compile(r"([a-zA-Z]+)\(([^)]*)\)")


def _local_name(element):
    return etree.QName(element).localname


def _svg_number(value):
    if value is None:
        return 0.0
    match = re.match(r"[-+0-9.eE]+", str(value).strip())
    return float(match.group(0)) if match else 0.0


def _parse_viewbox(root):
    viewbox = root.get("viewBox")
    if viewbox:
        x0, y0, width, height = map(float, viewbox.replace(",", " ").split())
        return x0, y0, width, height
    return 0.0, 0.0, _svg_number(root.get("width")), _svg_number(root.get("height"))


def _identity_matrix():
    return ((1.0, 0.0, 0.0), (0.0, 1.0, 0.0), (0.0, 0.0, 1.0))


def _matmul(a, b):
    return tuple(
        tuple(sum(a[i][k] * b[k][j] for k in range(3)) for j in range(3))
        for i in range(3)
    )


def _translation(tx=0.0, ty=0.0):
    return ((1.0, 0.0, tx), (0.0, 1.0, ty), (0.0, 0.0, 1.0))


def _scale(sx=1.0, sy=None):
    sy = sx if sy is None else sy
    return ((sx, 0.0, 0.0), (0.0, sy, 0.0), (0.0, 0.0, 1.0))


def _rotation(angle_deg, cx=0.0, cy=0.0):
    angle = angle_deg * np.pi / 180.0
    c = float(np.cos(angle))
    s = float(np.sin(angle))
    rotate = ((c, -s, 0.0), (s, c, 0.0), (0.0, 0.0, 1.0))
    return _matmul(_translation(cx, cy), _matmul(rotate, _translation(-cx, -cy)))


def _parse_transform(transform_text):
    if not transform_text:
        return _identity_matrix()

    matrix = _identity_matrix()
    for name, args_text in TRANSFORM_RE.findall(transform_text):
        args = [_svg_number(arg) for arg in re.split(r"[ ,]+", args_text.strip()) if arg]
        if name == "translate":
            tx = args[0] if args else 0.0
            ty = args[1] if len(args) > 1 else 0.0
            op = _translation(tx, ty)
        elif name == "scale":
            sx = args[0] if args else 1.0
            sy = args[1] if len(args) > 1 else None
            op = _scale(sx, sy)
        elif name == "matrix" and len(args) == 6:
            a, b, c, d, e, f = args
            op = ((a, c, e), (b, d, f), (0.0, 0.0, 1.0))
        elif name == "rotate":
            if len(args) == 1:
                op = _rotation(args[0])
            elif len(args) >= 3:
                op = _rotation(args[0], args[1], args[2])
            else:
                op = _identity_matrix()
        else:
            op = _identity_matrix()
        matrix = _matmul(matrix, op)
    return matrix


def _apply_matrix(matrix, x, y):
    return (
        matrix[0][0] * x + matrix[0][1] * y + matrix[0][2],
        matrix[1][0] * x + matrix[1][1] * y + matrix[1][2],
    )


def _composed_transform(element):
    matrices = []
    current = element
    while current is not None:
        matrices.append(_parse_transform(current.get("transform")))
        current = current.getparent()
    matrix = _identity_matrix()
    for op in reversed(matrices):
        matrix = _matmul(matrix, op)
    return matrix


def _all_top_level_groups(root):
    groups = []
    for child in root:
        if _local_name(child) == "g":
            groups.append(child)
    if not groups:
        raise ValueError("No top-level SVG group found")
    return groups


def _is_background_patch(child):
    if _local_name(child) != "g":
        return False
    child_id = child.get("id", "")
    if not child_id.startswith("patch_"):
        return False
    drawable_children = [grandchild for grandchild in child if _local_name(grandchild) in {"path", "rect", "polygon"}]
    if len(drawable_children) != 1:
        return False
    drawable = drawable_children[0]
    style = drawable.get("style", "")
    fill = drawable.get("fill")
    stroke = drawable.get("stroke")
    if fill is None:
        match = re.search(r"(?:^|;)\s*fill:\s*([^;]+)", style)
        fill = match.group(1).strip() if match else None
    if stroke is None:
        match = re.search(r"(?:^|;)\s*stroke:\s*([^;]+)", style)
        stroke = match.group(1).strip() if match else None
    has_fill = fill not in {None, "none", "None"}
    has_stroke = stroke not in {None, "none", "None"}
    return has_fill and not has_stroke


def _remove_patch_nodes(element):
    for child in list(element):
        if _is_background_patch(child):
            element.remove(child)
            continue
        _remove_patch_nodes(child)


def _gather_piece_anchors(root):
    anchors = {}
    for text in root.xpath(".//svg:text", namespaces=NS):
        label = "".join(text.itertext()).strip()
        if not label:
            continue
        x_values = (text.get("x") or "0").replace(",", " ").split()
        y_values = (text.get("y") or "0").replace(",", " ").split()
        x = _svg_number(x_values[0]) if x_values else 0.0
        y = _svg_number(y_values[0]) if y_values else 0.0
        abs_x, abs_y = _apply_matrix(_composed_transform(text), x, y)
        anchors.setdefault(label, []).append((abs_x, abs_y))
    return anchors


def _rename_ids(elements, prefix):
    id_map = {}
    for element in elements:
        for node in element.iter():
            node_id = node.get("id")
            if node_id:
                id_map[node_id] = f"{prefix}_{node_id}"

    for element in elements:
        for node in element.iter():
            node_id = node.get("id")
            if node_id:
                node.set("id", id_map[node_id])
            for attr_name, attr_value in list(node.attrib.items()):
                updated = attr_value
                for old_id, new_id in id_map.items():
                    updated = updated.replace(f"url(#{old_id})", f"url(#{new_id})")
                    if updated == f"#{old_id}":
                        updated = f"#{new_id}"
                if updated != attr_value:
                    node.set(attr_name, updated)
    return id_map


def _panel_container(group):
    children = list(group)
    if len(children) == 1:
        only_child = children[0]
        texts = [("".join(text.itertext()).strip()) for text in only_child.xpath('.//svg:text', namespaces=NS)]
        labels = [text for text in texts if TEXT_LABEL_RE.match(text)]
        if labels:
            return only_child
    return group


def _panel_slices(group):
    container = _panel_container(group)
    children = list(container)
    label_positions = []
    for idx, child in enumerate(children):
        texts = [("".join(text.itertext()).strip()) for text in child.xpath('.//svg:text', namespaces=NS)]
        labels = [text for text in texts if TEXT_LABEL_RE.match(text)]
        if labels:
            label_positions.append((labels[0], idx))

    panel_ranges = {}
    for pos, (label, start_idx) in enumerate(label_positions):
        end_idx = label_positions[pos + 1][1] if pos + 1 < len(label_positions) else len(children)
        panel_ranges[label] = (start_idx, end_idx)
    return container, panel_ranges


def drop_panels(piece, panel_labels):
    panel_labels = tuple(panel_labels)
    if not panel_labels:
        return piece

    group = _deepcopy(piece["group"])
    container, panel_ranges = _panel_slices(group)
    children = list(container)
    remove_indices = set()
    for label in panel_labels:
        if label not in panel_ranges:
            raise KeyError(f"Panel label {label!r} not found in {piece['path'].name}")
        start_idx, end_idx = panel_ranges[label]
        remove_indices.update(range(start_idx, end_idx))

    for idx in sorted(remove_indices, reverse=True):
        container.remove(children[idx])

    return {
        **piece,
        "group": group,
        "anchors": _gather_piece_anchors(group),
    }


def load_svg_piece(svg_path):
    svg_path = Path(svg_path)
    parser = etree.XMLParser(huge_tree=True)
    root = etree.parse(str(svg_path), parser).getroot()
    group = etree.Element(f"{{{SVG_NS}}}g")
    for top_group in _all_top_level_groups(root):
        group.append(_deepcopy(top_group))
    defs_nodes = [_deepcopy(node) for defs in root.xpath("./svg:defs", namespaces=NS) for node in defs]
    _remove_patch_nodes(group)
    anchors = _gather_piece_anchors(group)
    x0, y0, width, height = _parse_viewbox(root)
    return {
        "path": svg_path,
        "group": group,
        "defs": defs_nodes,
        "anchors": anchors,
        "viewbox": (x0, y0, width, height),
        "width": width,
        "height": height,
    }


def get_anchor(piece, anchor_text, occurrence=0):
    matches = piece["anchors"].get(anchor_text, [])
    if len(matches) <= occurrence:
        raise KeyError(f"Anchor {anchor_text!r} not found in {piece['path'].name}")
    return matches[occurrence]


def make_svg_root(width, height, x0=0.0, y0=0.0):
    root = etree.Element(
        f"{{{SVG_NS}}}svg",
        nsmap={None: SVG_NS, "xlink": XLINK_NS},
    )
    root.set("version", "1.1")
    root.set("width", f"{width}pt")
    root.set("height", f"{height}pt")
    root.set("viewBox", f"{x0} {y0} {width} {height}")
    defs = etree.SubElement(root, f"{{{SVG_NS}}}defs")
    return root, defs


def append_piece(root, defs_node, piece, placement_id, tx, ty):
    defs_nodes = [_deepcopy(node) for node in piece["defs"]]
    group = _deepcopy(piece["group"])
    _rename_ids(defs_nodes + [group], placement_id)
    for node in defs_nodes:
        defs_node.append(node)
    group.set("transform", f"translate({tx},{ty})")
    root.append(group)


def piece_box(piece, tx, ty):
    x0, y0, width, height = piece["viewbox"]
    return (tx + x0, ty + y0, tx + x0 + width, ty + y0 + height)


def _resolve_target_point(placements, align_to):
    target = placements[align_to["piece"]]
    target_anchor = get_anchor(target["anchor_piece_obj"], align_to["anchor"], align_to.get("occurrence", 0))
    return (
        target_anchor[0] + target["tx"],
        target_anchor[1] + target["ty"],
    )


def assemble_svg_recipe(recipe):
    source_dir = Path(recipe["source_dir"])
    output_path = source_dir / recipe["output_filename"]
    output_path.parent.mkdir(exist_ok=True)

    fig_width, fig_height = recipe["figsize"]
    canvas_width = fig_width * 72.0
    canvas_height = fig_height * 72.0
    root, defs_node = make_svg_root(canvas_width, canvas_height)

    main_piece_id = recipe["placements"][0]["piece"]
    deleted_from_main = []
    if recipe.get("delete_anchor_panels", True):
        for placement in recipe["placements"][1:]:
            align_to = placement.get("align_to")
            if not align_to:
                continue
            if align_to["piece"] == main_piece_id and TEXT_LABEL_RE.match(align_to["anchor"]):
                if align_to["anchor"] not in deleted_from_main:
                    deleted_from_main.append(align_to["anchor"])

    piece_ids = []
    for placement in recipe["placements"]:
        piece_id = placement["piece"]
        if piece_id not in piece_ids:
            piece_ids.append(piece_id)

    pieces = {}
    anchor_pieces = {}
    for piece_id in piece_ids:
        anchor_piece = load_svg_piece(source_dir / f"{piece_id}.svg")
        render_piece = anchor_piece
        if piece_id == main_piece_id and deleted_from_main:
            render_piece = drop_panels(anchor_piece, deleted_from_main)
        pieces[piece_id] = render_piece
        anchor_pieces[piece_id] = anchor_piece

    placements = {}
    visible_boxes = []
    for placement in recipe["placements"]:
        piece_id = placement["piece"]
        piece = pieces[piece_id]
        if not placements:
            tx, ty = 0.0, 0.0
        else:
            source_anchor = get_anchor(piece, placement["anchor"], placement.get("occurrence", 0))
            target_x, target_y = _resolve_target_point(placements, placement["align_to"])
            tx = target_x - source_anchor[0] + placement.get("dx", 0.0)
            ty = target_y - source_anchor[1] + placement.get("dy", 0.0)

        placements[piece_id] = {
            "piece_obj": piece,
            "anchor_piece_obj": anchor_pieces[piece_id],
            "tx": tx,
            "ty": ty,
        }
        append_piece(root, defs_node, piece, piece_id, tx, ty)
        visible_boxes.append(piece_box(piece, tx, ty))

    separator_boxes = []
    for separator in recipe.get("separators", []):
        left = placements[separator["left"]]
        right = placements[separator["right"]]
        left_box = piece_box(left["piece_obj"], left["tx"], left["ty"])
        right_box = piece_box(right["piece_obj"], right["tx"], right["ty"])
        x = (left_box[2] + right_box[0]) / 2.0 + separator.get("dx", 0.0)
        y0 = separator.get("y0", min(box[1] for box in visible_boxes)) + separator.get("top_trim", 0.0)
        y1 = separator.get("y1", max(box[3] for box in visible_boxes)) - separator.get("bottom_trim", 0.0)
        line = etree.SubElement(root, f"{{{SVG_NS}}}rect")
        line.set(
            "style",
            f"fill:{separator.get('color', '#828282')};fill-opacity:1;stroke:none;stroke-width:0",
        )
        line.set("x", str(x - separator.get("width", 1.0) / 2.0))
        line.set("y", str(y0))
        line.set("width", str(separator.get("width", 1.0)))
        line.set("height", str(y1 - y0))
        separator_boxes.append((x, y0, x, y1))

    all_boxes = visible_boxes + separator_boxes
    if all_boxes:
        min_x = min(box[0] for box in all_boxes)
        min_y = min(box[1] for box in all_boxes)
        max_x = max(box[2] for box in all_boxes)
        max_y = max(box[3] for box in all_boxes)
        width = max_x - min_x
        height = max_y - min_y
        root.set("viewBox", f"{min_x} {min_y} {width} {height}")
        root.set("width", f"{width}pt")
        root.set("height", f"{height}pt")

    etree.ElementTree(root).write(
        str(output_path),
        pretty_print=True,
        xml_declaration=True,
        encoding="utf-8",
    )
    print(f"Saved assembled SVG to {output_path}")
    return output_path


def run_recipe(recipe):
    output_path = assemble_svg_recipe(recipe)
    cleanup_svg_for_assembly(output_path)
    return output_path

In [10]:
a

NameError: name 'a' is not defined

## Model5

In [10]:
sizes = {
    "fig1": (9, 20.0),
    "fig2": (18, 16.0),
    "fig3": (18, 28.0),
}

In [11]:
# PIECES
for fig in (
    dict(
        piece="fig1_main",
        holo_pdbs = [
            "8jp0", 
            "8qtk", 
            "9fsj", 
            "6vvq", # "6vvq", "7e40",
            "7yg5", 
        ],
        rows=5, columns=1, figsize=sizes["fig1"],
    ),
    dict(
        piece="fig1_insert_row",
        holo_pdbs = ["6vvq", "7e40"],
        rows=5, columns=2, figsize=sizes["fig1"],
        single_item_rows={4}, empty_panel_slots=tuple(range(1,3*2+1)), label_start="d",
    ),
    dict(
        piece="fig1_tail_row",
        holo_pdbs = ["7yg5"],
        rows=5, columns=1, figsize=sizes["fig1"],
        single_item_rows={4}, empty_panel_slots=tuple(range(1,4+1)), label_start="f",
    ),
    dict(
        piece="fig2",
        holo_pdbs = [
            "8v81", "9ebs",
            "9oul", "6z1m",
            "9prs", "9o2m", 
            "7f8p", "8cgw",
        ],
        rows=4, columns=2, figsize=sizes["fig2"],
    ),
    dict(
        piece="si_main",
        holo_pdbs = [
            "8crc", "7gqu",
            "7qpn", "8f4s",
            "9dol", "7zpe",
            "8fpi", "7p2v",
            "22mj", # "22mj", "8uk6", "6s3a", "21du"
            "9mfs", #"8y6w",
            #"7v39", "7u4k",
        ],
        rows=7, columns=2, figsize=sizes["fig3"],
        empty_panel_slots=(10,)
    ),
    dict(
        piece="si_insert_row",
        holo_pdbs = ["22mj", "8uk6", "6s3a", "21du"],
        rows=7, columns=4, figsize=sizes["fig3"],
        single_item_rows={5}, empty_panel_slots=tuple(range(1,4*4+1)), label_start="i",
    ),
    dict(
        piece="si_tail_rows",
        holo_pdbs = [
            "9mfs", "8y6w",
            "7v39", "7u4k",
        ],
        rows=7, columns=2, figsize=sizes["fig3"],
        empty_panel_slots=tuple(range(1,5*2+1)), label_start="m",
    ),
):
    print("model5", fig["piece"])

    (OUTPUT_DIR / "model5").mkdir(exist_ok=True, parents=True)

    make_holo_apo_figure(
        model_name="model5",
        **{
            key: value for key, value in fig.items() if key != "piece"
        },
        output_filename=OUTPUT_DIR / "model5" / f'{fig["piece"]}.svg',
        show_panel_title=False,
    )

model5 fig1_main


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/fig1_main.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/fig1_main.svg
model5 fig1_insert_row


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/fig1_insert_row.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/fig1_insert_row.svg
model5 fig1_tail_row


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/fig1_tail_row.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/fig1_tail_row.svg
model5 fig2


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/fig2.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/fig2.svg
model5 si_main


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/si_main.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/si_main.svg
model5 si_insert_row


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/si_insert_row.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/si_insert_row.svg
model5 si_tail_rows


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/si_tail_rows.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/si_tail_rows.svg


In [12]:
# RECIPES
for recipe in (
    {
        "output_filename": "figure1_assembled.svg",
        "figsize": sizes["fig1"],
        "placements": [
            {"piece": "fig1_main"},
            {
                "piece": "fig1_insert_row",
                "anchor": "d)",
                "align_to": {"piece": "fig1_main", "anchor": "d)"},
            },
            {
                "piece": "fig1_tail_row",
                "anchor": "f)",
                "align_to": {"piece": "fig1_main", "anchor": "e)"},
            },
        ],
    },
    # {
    #     "output_filename": "figure2_assembled.svg",
    #     "figsize": sizes["fig2"],
    #     "placements": [
    #         {"piece": "fig2"},
    #     ],
    # },
    {
        "output_filename": "si_assembled.svg",
        "figsize": sizes["fig3"],
        "placements": [
            {"piece": "si_main"},
            {
                "piece": "si_insert_row",
                "anchor": "i)",
                "align_to": {"piece": "si_main", "anchor": "i)"},
            },
            {
                "piece": "si_tail_rows",
                "anchor": "m)",
                "align_to": {"piece": "si_main", "anchor": "j)"},
            },
        ],
    },
):
    run_recipe({
        **recipe,
        "source_dir": OUTPUT_DIR / "model5",
    })
    
    export_png(OUTPUT_DIR / "model5" / recipe["output_filename"])

Saved assembled SVG to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/figure1_assembled.svg


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/figure1_assembled.svg


Saved assembled SVG to /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/si_assembled.svg


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/model5_auto/si_assembled.svg


## Others

In [11]:
# 24.7 x 15
# 4.5 per column
total_height = 14
total_width = 24.7
column_width = 4.5

In [12]:
# PIECES

tmp = FIGURE_STYLE["item_label_fontsize"]
FIGURE_STYLE["item_label_fontsize"] = 12

for model in ("mefallosite",): #"passer_automl", "alloses", "allo", "allositepro",):
    for fig in (
        # # dict(
        # #     holo_pdbs = [
        # #         "8jp0", "8v81",                 "8f4s",
        # #                        # "9ebs", "7qpn",
        # #     ],
        # #     piece="blueprint",
        # #     rows=5, columns=5, figsize=(total_width, total_height),
        # #     empty_panel_slots=(3,4),
        # # ),
        dict(
            holo_pdbs = [
                "8jp0", 
                "8qtk", 
                "9fsj", 
                "6vvq",  # "6vvq", "7e40",
                "7yg5", 
            ],
            piece="fig1_main",
            rows=5, columns=1, figsize=(column_width, total_height),
        ),
        dict(
            piece="fig1_insert_row",
            holo_pdbs = ["6vvq", "7e40"],
            rows=5, columns=2, figsize=(column_width, total_height),
            single_item_rows={4}, empty_panel_slots=tuple(range(1,3*2+1)), label_start="d",
        ),
        dict(
            piece="fig1_tail_row",
            holo_pdbs = ["7yg5"],
            rows=5, columns=1, figsize=(column_width, total_height),
            single_item_rows={4}, empty_panel_slots=tuple(range(1,4+1)), label_start="f",
        ),
        dict(
            holo_pdbs = [
                "8v81", "9ebs",
                "9oul", "6z1m",
                "9prs", "9o2m", 
                "7f8p", "8cgw",
                "8crc", "7gqu", ### from 3
            ],
            piece="fig2",
            rows=5, columns=2, figsize=(column_width*2, total_height),
            label_start="g",
        ),
        dict(
            holo_pdbs = [
                ### "8crc", "7gqu",
                "7qpn", "8f4s",
                "9dol", "7zpe",
                "8fpi", "7p2v",
                "22mj", # "22mj", "8uk6", "6s3a", "21du"
                "9mfs", #"8y6w",
                #"7v39", "7u4k",
            ],
            piece="fig3_main",
            rows=6, columns=2, figsize=(column_width*2, total_height*1.05), 
            empty_panel_slots=(8,), label_start="q",
        ),
        dict(
            holo_pdbs = ["22mj", "8uk6", "6s3a", "21du"],
            piece="fig3_insert_row",
            rows=6, columns=4, figsize=(column_width*2, total_height*1.05),
            single_item_rows={4}, empty_panel_slots=tuple(range(1,3*4+1)), label_start="w"
        ),
        dict(
            piece="fig3_tail_rows",
            holo_pdbs = [
                "9mfs", "8y6w",
                "7v39", "7u4k",
            ],
            rows=7, columns=2, figsize=(column_width*2, total_height*1.05),
            empty_panel_slots=tuple(range(1,5*2+1)), label_start="aa",
        ),
    ):
        print(model, fig["piece"])
    
        (OUTPUT_DIR / model).mkdir(exist_ok=True, parents=True)
    
        make_holo_apo_figure(
            model_name=model,
            **{
                key: value for key, value in fig.items() if key != "piece"
            },
            output_filename=OUTPUT_DIR / model / f'{fig["piece"]}.svg',
            show_panel_title=False,
        )
    

FIGURE_STYLE["item_label_fontsize"] = tmp

mefallosite fig1_main


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig1_main.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig1_main.svg
mefallosite fig1_insert_row


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig1_insert_row.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig1_insert_row.svg
mefallosite fig1_tail_row


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig1_tail_row.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig1_tail_row.svg
mefallosite fig2


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig2.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig2.svg
mefallosite fig3_main


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig3_main.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig3_main.svg
mefallosite fig3_insert_row


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig3_insert_row.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig3_insert_row.svg
mefallosite fig3_tail_rows


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig3_tail_rows.svg
Saved figure to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/fig3_tail_rows.svg


In [13]:
# RECIPES

SECTION_GAP_PT = 42.0
SINGLE_COL_PT = column_width * 72.0
DOUBLE_COL_PT = 2 * column_width * 72.0
SECTION2_DX = SINGLE_COL_PT + SECTION_GAP_PT + (9.72 - 4.86)
SECTION3_DX = DOUBLE_COL_PT + SECTION_GAP_PT
SEPARATOR_STYLE = {
    "width": 0.4,
    "color": "#9a9a9a",
    "top_trim": 42.0,
    "bottom_trim": 30.0,
}


for model in ("mefallosite",): #"passer_automl", "alloses", "allo", "allositepro",):
    for recipe in (
        {
            "output_filename": "section1.svg",
            "figsize": (column_width, total_height),
            "placements": [
                {"piece": "fig1_main"},
                {
                    "piece": "fig1_insert_row",
                    "anchor": "d)",
                    "align_to": {"piece": "fig1_main", "anchor": "d)"},
                },
                {
                    "piece": "fig1_tail_row",
                    "anchor": "f)",
                    "align_to": {"piece": "fig1_main", "anchor": "e)"},
                },
            ],
        },
        {
            "output_filename": "section3.svg",
            "figsize": (2 * column_width, total_height*1.05),
            "placements": [
                {"piece": "fig3_main"},
                {
                    "piece": "fig3_insert_row",
                    "anchor": "w)",
                    "align_to": {"piece": "fig3_main", "anchor": "w)"},
                },
                {
                    "piece": "fig3_tail_rows",
                    "anchor": "aa)",
                    "align_to": {"piece": "fig3_main", "anchor": "x)"},
                },
            ],
        },
        {
            "output_filename": "assembled.svg",
            "figsize": (total_width, total_height),
            "delete_anchor_panels": False,
            "placements": [
                {"piece": "section1"},
                {
                    "piece": "fig2",
                    "anchor": "g)",
                    "align_to": {"piece": "section1", "anchor": "a)"},
                    "dx": SECTION2_DX,
                },
                {
                    "piece": "section3",
                    "anchor": "q)",
                    "align_to": {"piece": "fig2", "anchor": "g)"},
                    "dx": SECTION3_DX,
                },
            ],
            "separators": [
                {"left": "section1", "right": "fig2", **SEPARATOR_STYLE},
                {"left": "fig2", "right": "section3", **SEPARATOR_STYLE},
            ],
        },
    ):
        run_recipe({
            **recipe,
            "source_dir": OUTPUT_DIR / model,
            "output_filename": recipe["output_filename"].replace("assembled.svg", f"{model}_assembled.svg"),
        })

    export_png(OUTPUT_DIR / model / f"{model}_assembled.svg")

Saved assembled SVG to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/section1.svg


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/section1.svg


Saved assembled SVG to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/section3.svg


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/section3.svg


Saved assembled SVG to /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/mefallosite_assembled.svg


Cleaned /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/mefallosite_assembled.svg


Exported /home/fnerin/Desktop/AlloPockets/models/figure/mefallosite/mefallosite_assembled.png


### AllositePro

In [15]:
# 24.7 x 15
# 4.5 per column
total_height = 14
total_width = 24.7
column_width = 4.5

In [ ]:
tmp = FIGURE_STYLE["item_label_fontsize"]
FIGURE_STYLE["item_label_fontsize"] = 12

for model in ("passer_automl", "alloses",): #"allo",):# # "allositepro",
    for fig in (
        # # dict(
        # #     holo_pdbs = [
        # #         "8jp0", "8v81",                 "8f4s",
        # #                        # "9ebs", "7qpn",
        # #     ],
        # #     piece="blueprint",
        # #     rows=5, columns=5, figsize=(total_width, total_height),
        # #     empty_panel_slots=(3,4),
        # # ),
        dict(
            holo_pdbs = [
                "8jp0", 
                ### "8qtk", # AllositePro missing
                "9fsj", 
                "6vvq", # "7e40" # AllositePro missing
                "7yg5", 
            ],
            piece="fig1_main",
            rows=5, columns=1, figsize=(column_width, total_height),
        ),
dict(
            holo_pdbs = [
                "8jp0", 
                ### "8qtk", # AllositePro missing
                "9fsj", 
                "6vvq", # "7e40" # AllositePro missing
                "7yg5",
            ],
            output_filename="1_2.svg",
            rows=5, columns=1, figsize=(column_width, total_height),
            empty_panel_slots=(2,),



        
        dict(
            piece="fig1_insert_row",
            holo_pdbs = ["6vvq", "7e40"],
            rows=5, columns=2, figsize=(column_width, total_height),
            single_item_rows={4}, empty_panel_slots=tuple(range(1,3*2+1)), label_start="d",
        ),
        dict(
            piece="fig1_tail_row",
            holo_pdbs = ["7yg5"],
            rows=5, columns=1, figsize=(column_width, total_height),
            single_item_rows={4}, empty_panel_slots=tuple(range(1,4+1)), label_start="f",
        ),
        dict(
            holo_pdbs = [
                "8v81", "9ebs",
                "9oul", "6z1m",
                "9prs", "9o2m", 
                "7f8p", "8cgw",
                "8crc", "7gqu", ### from 3
            ],
            piece="fig2",
            rows=5, columns=2, figsize=(column_width*2, total_height),
            label_start="g",
        ),
        dict(
            holo_pdbs = [
                ### "8crc", "7gqu",
                "7qpn", "8f4s",
                "9dol", "7zpe",
                "8fpi", "7p2v",
                "22mj", # "22mj", "8uk6", "6s3a", "21du"
                "9mfs", #"8y6w",
                #"7v39", "7u4k",
            ],
            piece="fig3_main",
            rows=6, columns=2, figsize=(column_width*2, total_height*1.05), 
            empty_panel_slots=(8,), label_start="q",
        ),
        dict(
            holo_pdbs = ["22mj", "8uk6", "6s3a", "21du"],
            piece="fig3_insert_row",
            rows=6, columns=4, figsize=(column_width*2, total_height*1.05),
            single_item_rows={4}, empty_panel_slots=tuple(range(1,3*4+1)), label_start="w"
        ),
        dict(
            piece="fig3_tail_rows",
            holo_pdbs = [
                "9mfs", "8y6w",
                "7v39", "7u4k",
            ],
            rows=7, columns=2, figsize=(column_width*2, total_height*1.05),
            empty_panel_slots=tuple(range(1,5*2+1)), label_start="aa",
        ),
    ):
        print(model, fig["piece"])
    
        (OUTPUT_DIR / model).mkdir(exist_ok=True, parents=True)
    
        make_holo_apo_figure(
            model_name=model,
            **{
                key: value for key, value in fig.items() if key != "piece"
            },
            output_filename=OUTPUT_DIR / model / f'{fig["piece"]}.svg',
            show_panel_title=False,
        )


FIGURE_STYLE["item_label_fontsize"] = tmp

In [ ]:
SECTION_GAP_PT = 42.0
SINGLE_COL_PT = column_width * 72.0
DOUBLE_COL_PT = 2 * column_width * 72.0
SECTION2_DX = SINGLE_COL_PT + SECTION_GAP_PT + (9.72 - 4.86)
SECTION3_DX = DOUBLE_COL_PT + SECTION_GAP_PT
SEPARATOR_STYLE = {
    "width": 0.4,
    "color": "#9a9a9a",
    "top_trim": 42.0,
    "bottom_trim": 30.0,
}


for model in ("passer_automl", "alloses",): #"allo",):# # "allositepro",
    for recipe in (
        {
            "output_filename": "section1.svg",
            "figsize": (column_width, total_height),
            "placements": [
                {"piece": "fig1_main"},
                {
                    "piece": "fig1_insert_row",
                    "anchor": "d)",
                    "align_to": {"piece": "fig1_main", "anchor": "d)"},
                },
                {
                    "piece": "fig1_tail_row",
                    "anchor": "f)",
                    "align_to": {"piece": "fig1_main", "anchor": "e)"},
                },
            ],
        },
        {
            "output_filename": "section3.svg",
            "figsize": (2 * column_width, total_height*1.05),
            "placements": [
                {"piece": "fig3_main"},
                {
                    "piece": "fig3_insert_row",
                    "anchor": "w)",
                    "align_to": {"piece": "fig3_main", "anchor": "w)"},
                },
                {
                    "piece": "fig3_tail_rows",
                    "anchor": "aa)",
                    "align_to": {"piece": "fig3_main", "anchor": "x)"},
                },
            ],
        },
        {
            "output_filename": "assembled.svg",
            "figsize": (total_width, total_height),
            "delete_anchor_panels": False,
            "placements": [
                {"piece": "section1"},
                {
                    "piece": "fig2",
                    "anchor": "g)",
                    "align_to": {"piece": "section1", "anchor": "a)"},
                    "dx": SECTION2_DX,
                },
                {
                    "piece": "section3",
                    "anchor": "q)",
                    "align_to": {"piece": "fig2", "anchor": "g)"},
                    "dx": SECTION3_DX,
                },
            ],
            "separators": [
                {"left": "section1", "right": "fig2", **SEPARATOR_STYLE},
                {"left": "fig2", "right": "section3", **SEPARATOR_STYLE},
            ],
        },
    ):
        run_recipe({
            **recipe,
            "source_dir": OUTPUT_DIR / model,
        })

